# FloodSeg: Exploratory Data Analysis & Chip Visualisation

This notebook inspects hand-labeled satellite chips from the **Sen1Floods11** dataset.

For each sample chip, we visualize:
1. **Sentinel-2 Optical True-Color (RGB):** Bands 4 (Red), 3 (Green), 2 (Blue) with 2%–98% percentile contrast stretch.
2. **Sentinel-1 SAR VV Polarization (dB):** Radar backscatter sensitive to surface roughness (calm water = dark).
3. **Sentinel-1 SAR VH Polarization (dB):** Cross-polarization sensitive to volumetric scattering and vegetation canopy.
4. **Ground-Truth Hand Label:** Expert annotations with `1 = Water` (Cyan), `0 = Not Water` (Dark Gray), `-1 = Invalid/No-data` (Yellow/Hatched).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import csv

from floodseg.data.dataset import Sen1Floods11Dataset

## 1. Load Split and Chips

In [ ]:
root_dir = Path("..") / "data"
split_file = root_dir / "processed" / "splits" / "official" / "flood_train_data.csv"

# Fallback if running notebook directly from notebooks directory
if not split_file.exists():
    split_file = Path("data/processed/splits/official/flood_train_data.csv")
    root_dir = Path("data")

dataset = Sen1Floods11Dataset(root_dir=root_dir, split_file=split_file, modality="fusion")
print(f"Loaded {len(dataset)} chips from {split_file.name}")

## 2. Visualization Utilities
We implement percentile contrast stretching for 12-bit / TOA Sentinel-2 optical data and dB clipping for Sentinel-1.

In [ ]:
def stretch_optical_rgb(s2_cube):
    """Extract RGB (Bands 4, 3, 2 -> indices 3, 2, 1) and apply 2%-98% stretch."""
    # Sen1Floods11 band ordering: 0:B1, 1:B2(Blue), 2:B3(Green), 3:B4(Red), ...
    rgb = s2_cube[:, :, [3, 2, 1]].copy()
    rgb = np.nan_to_num(rgb, nan=0.0)
    
    p2, p98 = np.percentile(rgb, (2, 98))
    if p98 > p2:
        rgb = (rgb - p2) / (p98 - p2)
    else:
        rgb = rgb / 10000.0
    return np.clip(rgb, 0.0, 1.0)

def clip_sar_db(sar_band, vmin=-25.0, vmax=0.0):
    """Clip SAR backscatter in dB to standard range."""
    clipped = np.clip(sar_band, vmin, vmax)
    return (clipped - vmin) / (vmax - vmin)

# Custom colormap for labels: -1: Gold/Yellow, 0: Dark Gray, 1: Vibrant Cyan
cmap_label = mcolors.ListedColormap(["#d4af37", "#2b2b2b", "#00e5ff"])
norm_label = mcolors.BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap_label.N)

## 3. Visualise 10 Random Chips

In [ ]:
np.random.seed(42)
num_samples = min(10, len(dataset))
indices = np.random.choice(len(dataset), size=num_samples, replace=False)

fig, axes = plt.subplots(num_samples, 4, figsize=(18, 4.2 * num_samples))
if num_samples == 1:
    axes = np.expand_dims(axes, 0)

col_titles = ["Sentinel-2 True Color (RGB)", "Sentinel-1 SAR VV (dB)", "Sentinel-1 SAR VH (dB)", "Hand Label"]
for col_idx, title in enumerate(col_titles):
    axes[0, col_idx].set_title(title, fontsize=14, fontweight="bold", pad=12)

for row_idx, chip_idx in enumerate(indices):
    sample_dict = dataset.samples[chip_idx]
    raw_fusion = dataset.get_raw_image(chip_idx)  # [H, W, 15] (2 S1 + 13 S2)
    label = dataset._read_label(sample_dict["label_path"])  # [H, W]
    
    s1_vv = raw_fusion[:, :, 0]
    s1_vh = raw_fusion[:, :, 1]
    s2_data = raw_fusion[:, :, 2:]
    
    # 1. Optical RGB
    rgb_vis = stretch_optical_rgb(s2_data)
    axes[row_idx, 0].imshow(rgb_vis)
    axes[row_idx, 0].set_ylabel(f"{sample_dict['chip_id']}", fontsize=11, fontweight="semibold")
    
    # 2. SAR VV
    vv_vis = clip_sar_db(s1_vv)
    axes[row_idx, 1].imshow(vv_vis, cmap="gray")
    
    # 3. SAR VH
    vh_vis = clip_sar_db(s1_vh)
    axes[row_idx, 2].imshow(vh_vis, cmap="gray")
    
    # 4. Label
    im = axes[row_idx, 3].imshow(label, cmap=cmap_label, norm=norm_label)
    
    for ax in axes[row_idx]:
        ax.set_xticks([])
        ax.set_yticks([])

# Legend for ground truth
legend_patches = [
    plt.Rectangle((0,0),1,1, color="#00e5ff", label="Water (1)"),
    plt.Rectangle((0,0),1,1, color="#2b2b2b", label="Not Water (0)"),
    plt.Rectangle((0,0),1,1, color="#d4af37", label="Invalid / No-data (-1)"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.01), fontsize=13)
plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.savefig("notebooks/sample_chips_visualization.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Class Imbalance Analysis across Sample Chips

In [ ]:
total_water = 0
total_land = 0
total_invalid = 0

for chip_idx in indices:
    label = dataset._read_label(dataset.samples[chip_idx]["label_path"])
    total_water += np.sum(label == 1)
    total_land += np.sum(label == 0)
    total_invalid += np.sum(label == -1)

total_pixels = total_water + total_land + total_invalid
print(f"Total Analyzed Pixels: {total_pixels:,}")
print(f"  - Water: {total_water:,} ({100 * total_water / total_pixels:.2f}%)")
print(f"  - Land:  {total_land:,} ({100 * total_land / total_pixels:.2f}%)")
print(f"  - Invalid: {total_invalid:,} ({100 * total_invalid / total_pixels:.2f}%)")